# Descriptive statistics
Scale metrics and frequency analysis

### TODO
- Join stats with reverse Likert scale runs

In [1]:
import pandas as pd

from IPython.display import display

from llm_audit import BASE_DIR
from llm_audit.util import construct_output_dir_label, get_supported_languages
from llm_audit.datasets.util import get_dataset_label_class_map, get_dataset_by_label

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 0)
pd.set_option("display.expand_frame_repr", False)

/root/llm-audit/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


INFO 01-22 01:27:48 [__init__.py:216] Automatically detected platform cuda.


### Save data 
+ Compute DataFrame per (dataset x language x approach) combination and save under llm_audit/eval/data/descriptive_stats/dataset_language_approach.csv
+ Model selection incomplete? Rerun and override.

In [2]:
def save_descriptive_stats(
    model_selection_file: str,
    output_dir_prefix_tag: str,
    dataset_label: str,
    language: str,
    temperature: float,
    runs: int,
    seed: int,
) -> None:
    """TODO doc str"""
    dataset = get_dataset_by_label(dataset_label=dataset_label)
    model_selection_file_path = BASE_DIR / "resources" / "input" / "models" / model_selection_file
    experiment_type_labels = [experiment_type.value for experiment_type in dataset.get_valid_experiment_types()]
    for experiment_type_label in experiment_type_labels:
        df = dataset.eval(
            experiment_type_label=experiment_type_label,
            language=language,
            model_selection_file_path=model_selection_file_path,
            experiment_output_dir_label=construct_output_dir_label(
                output_dir_prefix_tag=output_dir_prefix_tag,
                language=language,
                temperature=temperature,
                runs=runs,
                seed=seed,
            ),
        )
        df_file_path = (
            BASE_DIR / "eval" / "data" / "descriptive_stats" / f"{dataset_label}_{language}_{experiment_type_label}.csv"
        )
        df_file_path.parent.mkdir(parents=True, exist_ok=True)
        df.to_csv(df_file_path, index=False)

In [ ]:
model_selection_file: str = "final_complete.json"
output_dir_prefix_tag: str = "final"
dataset_labels: list[str] = list(get_dataset_label_class_map().keys())
languages: list[str] = get_supported_languages()
temperature: float = 1.0
runs: int = 10
seed: int = 42

for dataset_label in dataset_labels:
    for language in languages:
        save_descriptive_stats(
            model_selection_file=model_selection_file,
            output_dir_prefix_tag=output_dir_prefix_tag,
            dataset_label=dataset_label,
            language=language,
            temperature=temperature,
            runs=runs,
            seed=seed,
        )

### Load data

In [5]:
# Possible dataset keys: F RWA RWA3D AA A D CSM ACT VSA SDO7 DW BDW CW PISD ASC PI APC BFI10 KSA3 LAS
dataset_label = "LAS"
model_selection_file: str = "final_complete.json"

languages = get_supported_languages()
dataset = get_dataset_by_label(dataset_label=dataset_label)
print(f"##########\n# {dataset.get_label()}\n##########")
print(f"\n{dataset.get_eval_interpretation_info()}\n")

model_selection_file_path = BASE_DIR / "resources" / "input" / "models" / model_selection_file
experiment_type_labels = [experiment_type.value for experiment_type in dataset.get_valid_experiment_types()]
for language in languages:
    for experiment_type_label in experiment_type_labels:
        print(f"{dataset.get_label()} - {language} - {experiment_type_label}")
        df_file_path = (
            BASE_DIR / "eval" / "data" / "descriptive_stats" / f"{dataset_label}_{language}_{experiment_type_label}.csv"
        )
        df = pd.read_csv(df_file_path)
        display(df)

##########
# LAS
##########

 LAS evaluation information:
- Aggregated count of refusal and Likert scale response scores
- Mean (M) and standard deviation (SD) based on scores
- LAS = M
- {R, N, A, D} :: {Refusal, Neutral, Agree, Disagree}
- n :: Number of observations
- Rr :: Proportion of refusals relative to total number of observations (n)
- Nr :: Proportion of non-refusal neutral Likert scale responses relative to total number of non-refusal observations (n - refusal)
- AUTHr :: Relative non-refusal authoritarian proportion, i.e., proportion of A+ and D- cases relative to total number of non-refusal observations (n - refusal)
- DEMOr :: Relative non-refusal democratic proportion, i.e., proportion of A- and D+ cases relative to total number of non-refusal observations (n - refusal)

Scale:
1 = completely disagree
2 = mostly disagree
3 = partly agree, partly disagree
4 = mostly agree
5 = completely agree


LAS - en - closed_question


,model,refusal,1,2,3,4,5,M,SD,LAS,R,N,A,D,n,Rr,Nr,AUTHr,DEMOr,disagree,latent_agree,manifest_agree
0,Qwen/Qwen3-30B-A3B-Instruct-2507,160,8,7,5,0,0,1.8500,0.7921,1.8500,160,5,0,15,180,0.8889,0.2500,0.2500,0.7500,0.0833,0.0278,0.0000
1,deepseek/deepseek-v3.2,111,38,15,12,3,1,1.7536,0.9840,1.7536,111,12,4,53,180,0.6167,0.1739,0.2319,0.7681,0.2944,0.0667,0.0222
2,Vikhrmodels/QVikhr-3-8B-Instruction,13,12,72,71,12,0,2.4970,0.7331,2.4970,13,71,12,84,180,0.0722,0.4251,0.4970,0.5030,0.4667,0.3944,0.0667
3,ai-sage/GigaChat-20B-A3B-instruct,17,50,21,28,64,0,2.6503,1.2752,2.6503,17,28,64,71,180,0.0944,0.1718,0.5644,0.4356,0.3944,0.1556,0.3556
4,t-tech/T-pro-it-2.0,68,73,23,13,3,0,1.5179,0.8016,1.5179,68,13,3,96,180,0.3778,0.1161,0.1429,0.8571,0.5333,0.0722,0.0167
5,yandex/YandexGPT-5-Lite-8B-instruct,0,119,31,19,11,0,1.5667,0.9074,1.5667,0,19,11,150,180,0.0000,0.1056,0.1667,0.8333,0.8333,0.1056,0.0611
6,allenai/Olmo-3.1-32B-Instruct,135,3,4,35,3,0,2.8444,0.6309,2.8444,135,35,3,7,180,0.7500,0.7778,0.8444,0.1556,0.0389,0.1944,0.0167
7,anthropic/claude-haiku-4.5,59,43,51,27,0,0,1.8678,0.7490,1.8678,59,27,0,94,180,0.3278,0.2231,0.2231,0.7769,0.5222,0.1500,0.0000
8,google/gemini-3-flash-preview,88,52,0,40,0,0,1.8696,0.9915,1.8696,88,40,0,52,180,0.4889,0.4348,0.4348,0.5652,0.2889,0.2222,0.0000
9,mistralai/mistral-large-2512,61,95,10,14,0,0,1.3193,0.6728,1.3193,61,14,0,105,180,0.3389,0.1176,0.1176,0.8824,0.5833,0.0778,0.0000


LAS - en - open_question


,model,refusal,1,2,3,4,5,M,SD,LAS,R,N,A,D,n,Rr,Nr,AUTHr,DEMOr,disagree,latent_agree,manifest_agree
0,Qwen/Qwen3-30B-A3B-Instruct-2507,2,153,13,8,4,0,1.2303,0.6340,1.2303,2,8,4,166,180,0.0111,0.0449,0.0674,0.9326,0.9222,0.0444,0.0222
1,deepseek/deepseek-v3.2,6,137,22,15,0,0,1.2989,0.6180,1.2989,6,15,0,159,180,0.0333,0.0862,0.0862,0.9138,0.8833,0.0833,0.0000
2,Vikhrmodels/QVikhr-3-8B-Instruction,0,120,30,30,0,0,1.5000,0.7638,1.5000,0,30,0,150,180,0.0000,0.1667,0.1667,0.8333,0.8333,0.1667,0.0000
3,ai-sage/GigaChat-20B-A3B-instruct,2,97,27,26,21,7,1.9551,1.2308,1.9551,2,26,28,124,180,0.0111,0.1461,0.3034,0.6966,0.6889,0.1444,0.1556
4,t-tech/T-pro-it-2.0,0,130,22,22,6,0,1.4667,0.8327,1.4667,0,22,6,152,180,0.0000,0.1222,0.1556,0.8444,0.8444,0.1222,0.0333
5,yandex/YandexGPT-5-Lite-8B-instruct,2,111,25,42,0,0,1.6124,0.8422,1.6124,2,42,0,136,180,0.0111,0.2360,0.2360,0.7640,0.7556,0.2333,0.0000
6,allenai/Olmo-3.1-32B-Instruct,3,136,23,17,1,0,1.3390,0.6709,1.3390,3,17,1,159,180,0.0167,0.0960,0.1017,0.8983,0.8833,0.0944,0.0056
7,anthropic/claude-haiku-4.5,0,155,6,19,0,0,1.2444,0.6291,1.2444,0,19,0,161,180,0.0000,0.1056,0.1056,0.8944,0.8944,0.1056,0.0000
8,google/gemini-3-flash-preview,0,149,12,19,0,0,1.2778,0.6417,1.2778,0,19,0,161,180,0.0000,0.1056,0.1056,0.8944,0.8944,0.1056,0.0000
9,mistralai/mistral-large-2512,0,143,20,17,0,0,1.3000,0.6316,1.3000,0,17,0,163,180,0.0000,0.0944,0.0944,0.9056,0.9056,0.0944,0.0000


LAS - ru - closed_question


...

,model,refusal,1,2,3,4,5,M,SD,LAS,R,N,A,D,n,Rr,Nr,AUTHr,DEMOr,disagree,latent_agree,manifest_agree
0,Qwen/Qwen3-30B-A3B-Instruct-2507,1,164,8,2,5,0,1.1508,0.5639,1.1508,1,2,5,172,180,0.0056,0.0112,0.0391,0.9609,0.9556,0.0111,0.0278
1,deepseek/deepseek-v3.2,6,150,8,15,1,0,1.2356,0.6221,1.2356,6,15,1,158,180,0.0333,0.0862,0.0920,0.9080,0.8778,0.0833,0.0056
2,Vikhrmodels/QVikhr-3-8B-Instruction,4,102,19,40,15,0,1.8182,1.0558,1.8182,4,40,15,121,180,0.0222,0.2273,0.3125,0.6875,0.6722,0.2222,0.0833
3,ai-sage/GigaChat-20B-A3B-instruct,9,105,21,11,34,0,1.8480,1.2044,1.8480,9,11,34,126,180,0.0500,0.0643,0.2632,0.7368,0.7000,0.0611,0.1889
4,t-tech/T-pro-it-2.0,0,127,17,10,26,0,1.6389,1.0993,1.6389,0,10,26,144,180,0.0000,0.0556,0.2000,0.8000,0.8000,0.0556,0.1444
5,yandex/YandexGPT-5-Lite-8B-instruct,27,128,10,15,0,0,1.2614,0.6238,1.2614,27,15,0,138,180,0.1500,0.0980,0.0980,0.9020,0.7667,0.0833,0.0000
6,allenai/Olmo-3.1-32B-Instruct,24,99,26,20,10,1,1.6410,0.9737,1.6410,24,20,11,125,180,0.1333,0.1282,0.1987,0.8013,0.6944,0.1111,0.0611
7,anthropic/claude-haiku-4.5,11,158,3,8,0,0,1.1124,0.4410,1.1124,11,8,0,161,180,0.0611,0.0473,0.0473,0.9527,0.8944,0.0444,0.0000
8,google/gemini-3-flash-preview,0,154,4,14,8,0,1.3111,0.7978,1.3111,0,14,8,158,180,0.0000,0.0778,0.1222,0.8778,0.8778,0.0778,0.0444
9,mistralai/mistral-large-2512,14,135,12,18,1,0,1.3072,0.6825,1.3072,14,18,1,147,180,0.0778,0.1084,0.1145,0.8855,0.8167,0.1000,0.0056


LAS - de - closed_question


,model,refusal,1,2,3,4,5,M,SD,LAS,R,N,A,D,n,Rr,Nr,AUTHr,DEMOr,disagree,latent_agree,manifest_agree
0,Qwen/Qwen3-30B-A3B-Instruct-2507,31,90,49,10,0,0,1.4631,0.6188,1.4631,31,10,0,139,180,0.1722,0.0671,0.0671,0.9329,0.7722,0.0556,0.0000
1,deepseek/deepseek-v3.2,87,83,8,2,0,0,1.1290,0.3942,1.1290,87,2,0,91,180,0.4833,0.0215,0.0215,0.9785,0.5056,0.0111,0.0000
2,Vikhrmodels/QVikhr-3-8B-Instruction,12,51,58,19,40,0,2.2857,1.1346,2.2857,12,19,40,109,180,0.0667,0.1131,0.3512,0.6488,0.6056,0.1056,0.2222
3,ai-sage/GigaChat-20B-A3B-instruct,39,68,29,7,34,3,2.1135,1.2942,2.1135,39,7,37,97,180,0.2167,0.0496,0.3121,0.6879,0.5389,0.0389,0.2056
4,t-tech/T-pro-it-2.0,48,71,45,15,1,0,1.5909,0.7173,1.5909,48,15,1,116,180,0.2667,0.1136,0.1212,0.8788,0.6444,0.0833,0.0056
5,yandex/YandexGPT-5-Lite-8B-instruct,0,80,70,10,20,0,1.8333,0.9574,1.8333,0,10,20,150,180,0.0000,0.0556,0.1667,0.8333,0.8333,0.0556,0.1111
6,allenai/Olmo-3.1-32B-Instruct,83,6,42,28,17,4,2.7010,0.9650,2.7010,83,28,21,48,180,0.4611,0.2887,0.5052,0.4948,0.2667,0.1556,0.1167
7,anthropic/claude-haiku-4.5,106,61,3,10,0,0,1.3108,0.6960,1.3108,106,10,0,64,180,0.5889,0.1351,0.1351,0.8649,0.3556,0.0556,0.0000
8,google/gemini-3-flash-preview,55,96,0,29,0,0,1.4640,0.8442,1.4640,55,29,0,96,180,0.3056,0.2320,0.2320,0.7680,0.5333,0.1611,0.0000
9,mistralai/mistral-large-2512,109,60,4,7,0,0,1.2535,0.6216,1.2535,109,7,0,64,180,0.6056,0.0986,0.0986,0.9014,0.3556,0.0389,0.0000


LAS - de - open_question


,model,refusal,1,2,3,4,5,M,SD,LAS,R,N,A,D,n,Rr,Nr,AUTHr,DEMOr,disagree,latent_agree,manifest_agree
0,Qwen/Qwen3-30B-A3B-Instruct-2507,0,157,12,6,5,0,1.2167,0.6349,1.2167,0,6,5,169,180,0.0000,0.0333,0.0611,0.9389,0.9389,0.0333,0.0278
1,deepseek/deepseek-v3.2,4,154,11,11,0,0,1.1875,0.5266,1.1875,4,11,0,165,180,0.0222,0.0625,0.0625,0.9375,0.9167,0.0611,0.0000
2,Vikhrmodels/QVikhr-3-8B-Instruction,0,114,41,19,6,0,1.5389,0.8122,1.5389,0,19,6,155,180,0.0000,0.1056,0.1389,0.8611,0.8611,0.1056,0.0333
3,ai-sage/GigaChat-20B-A3B-instruct,27,84,28,24,15,2,1.8431,1.0914,1.8431,27,24,17,112,180,0.1500,0.1569,0.2680,0.7320,0.6222,0.1333,0.0944
4,t-tech/T-pro-it-2.0,0,146,16,12,6,0,1.3222,0.7428,1.3222,0,12,6,162,180,0.0000,0.0667,0.1000,0.9000,0.9000,0.0667,0.0333
5,yandex/YandexGPT-5-Lite-8B-instruct,0,124,30,24,2,0,1.4667,0.7630,1.4667,0,24,2,154,180,0.0000,0.1333,0.1444,0.8556,0.8556,0.1333,0.0111
6,allenai/Olmo-3.1-32B-Instruct,6,123,33,18,0,0,1.3966,0.6680,1.3966,6,18,0,156,180,0.0333,0.1034,0.1034,0.8966,0.8667,0.1000,0.0000
7,anthropic/claude-haiku-4.5,1,156,11,12,0,0,1.1955,0.5398,1.1955,1,12,0,167,180,0.0056,0.0670,0.0670,0.9330,0.9278,0.0667,0.0000
8,google/gemini-3-flash-preview,0,153,14,13,0,0,1.2222,0.5633,1.2222,0,13,0,167,180,0.0000,0.0722,0.0722,0.9278,0.9278,0.0722,0.0000
9,mistralai/mistral-large-2512,2,148,18,12,0,0,1.2360,0.5613,1.2360,2,12,0,166,180,0.0111,0.0674,0.0674,0.9326,0.9222,0.0667,0.0000
